# AQI prediction and multi-day early warning system ? India

A reproducible **pure machine-learning** study notebook for:

1. an eight-model AQI regression benchmark on the five-city pollutant dataset;
2. t+1, t+2, and t+3 AQI trajectory forecasting; and
3. cost-sensitive nationwide early warnings for AQI >200 and AQI >300.

No neural networks are used. All evaluation is chronological. Dynamic predictors at issue date *t* are never sourced from after *t*.

> **Data provenance:** The workspace AQI input is a Kaggle dataset, documented as an area-level daily AQI aggregate. Do not describe it as a direct CPCB extract unless that provenance is independently verified. NASA POWER is historical reanalysis?not a live weather forecast?so the early-warning experiment is a historical back-test. A real deployment must substitute weather forecasts available at issue time.


## Install packages if needed

Run this cell once if the kernel is missing a dependency, then restart it. LightGBM and CatBoost are required to complete the requested eight-model comparison.


In [ ]:
# %pip install -U pandas numpy scikit-learn xgboost lightgbm catboost matplotlib seaborn joblib


## 1. Imports and reproducibility configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, precision_score, r2_score,
    recall_score, roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

# Explicit workspace location. This avoids Jupyter using its launch directory as the project root.
PROJECT_ROOT = Path(r"C:\Users\cdipi\Downloads\aqi_for india")
assert (PROJECT_ROOT / "data" / "processed").exists(), f"Project data not found at {PROJECT_ROOT}"

METRO_PATH = PROJECT_ROOT / "data" / "processed" / "aqi_prediction_metro.csv"
FULL_PATH = PROJECT_ROOT / "data" / "processed" / "aqi_prediction_full.csv"
REPORT_DIR = PROJECT_ROOT / "reports" / "research_ml"
MODEL_DIR = PROJECT_ROOT / "models" / "research_ml"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
FORECAST_HORIZONS = [1, 2, 3]
EVENT_THRESHOLDS = [200, 300]
# Set True while developing; use False for the paper results.
QUICK_RUN = False
TREE_ESTIMATORS = 100 if QUICK_RUN else 300

# Metro experiment: origins before this date train; following dates are untouched test origins.
METRO_TEST_START = pd.Timestamp("2024-07-01")
# Nationwide event experiment uses independent validation dates to choose alert thresholds.
EW_TRAIN_END = pd.Timestamp("2024-04-30")
EW_VALID_END = pd.Timestamp("2024-10-31")
EW_TEST_END = pd.Timestamp("2025-04-27")
TARGET_RECALL = 0.80
COST_MULTIPLIER = 2.0

assert METRO_PATH.exists(), METRO_PATH
assert FULL_PATH.exists(), FULL_PATH
print(f"Project root: {PROJECT_ROOT}")
print(f"LightGBM available: {LIGHTGBM_AVAILABLE}; CatBoost available: {CATBOOST_AVAILABLE}")


## 2. Leakage-safe metro forecasting table

At issue date *t*, current pollutant and weather observations are allowed, along with their historical lags. The response is AQI at *t+h*. `air_quality_status`, `prominent_pollutants`, and all information from *t+h* are excluded. Rows with date gaps do not receive a false lag or future label.


In [ ]:
POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"]
WEATHER = [
    "temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
    "relative_humidity_2m_mean", "precipitation_sum", "wind_speed_2m_mean",
    "wind_speed_2m_max", "surface_pressure_mean", "shortwave_radiation_sum",
]

def make_metro_forecast_table(path: Path) -> tuple[pd.DataFrame, list[str], list[str]]:
    data = pd.read_csv(path, parse_dates=["date"]).copy()
    required = {"date", "area", "aqi_value", *POLLUTANTS, *WEATHER}
    missing = required - set(data.columns)
    assert not missing, f"Metro data missing columns: {missing}"
    data = data.sort_values(["area", "date"]).drop_duplicates(["area", "date"]).reset_index(drop=True)
    group = data.groupby("area", sort=False)

    # Current values are observed by forecast issue time t; lags are t-1 and t-2.
    data = data.rename(columns={"aqi_value": "aqi_at_issue"})
    group = data.groupby("area", sort=False)
    source_cols = ["aqi_at_issue", *POLLUTANTS, *WEATHER]
    for col in source_cols:
        for lag in (1, 2):
            name = f"{col}_lag_{lag}"
            data[name] = group[col].shift(lag)
            source_date = group["date"].shift(lag)
            data.loc[(data.date - source_date).dt.days.ne(lag), name] = np.nan

    data["wind_speed_delta_1d"] = data["wind_speed_2m_mean"] - data["wind_speed_2m_mean_lag_1"]
    data["aqi_change_1d"] = data["aqi_at_issue"] - data["aqi_at_issue_lag_1"]
    data["month_sin"] = np.sin(2 * np.pi * data.date.dt.month / 12)
    data["month_cos"] = np.cos(2 * np.pi * data.date.dt.month / 12)
    data["doy_sin"] = np.sin(2 * np.pi * data.date.dt.dayofyear / 365.25)
    data["doy_cos"] = np.cos(2 * np.pi * data.date.dt.dayofyear / 365.25)
    data["is_weekend"] = data.date.dt.dayofweek.isin([5, 6]).astype(int)

    # Shifted targets; a target is valid only if its calendar date is exactly h days ahead.
    for h in FORECAST_HORIZONS:
        target = group["aqi_at_issue"].shift(-h)
        target_date = group["date"].shift(-h)
        data[f"target_aqi_h{h}"] = target.where((target_date - data.date).dt.days.eq(h))

    numeric = source_cols + [f"{c}_lag_{lag}" for c in source_cols for lag in (1, 2)]
    numeric += ["wind_speed_delta_1d", "aqi_change_1d", "month_sin", "month_cos", "doy_sin", "doy_cos", "is_weekend"]
    categorical = ["area"]
    return data, numeric, categorical

metro, METRO_NUMERIC, METRO_CATEGORICAL = make_metro_forecast_table(METRO_PATH)
print(f"Metro origins: {len(metro):,}; cities: {metro.area.nunique()}; dates: {metro.date.min():%F} to {metro.date.max():%F}")
print(f"Predictors: {len(METRO_NUMERIC)} numeric + {len(METRO_CATEGORICAL)} categorical")
display(metro[["date", "area", "aqi_at_issue", "target_aqi_h1", "target_aqi_h2", "target_aqi_h3"]].head())


## 3. Preprocessing and common metrics

In [ ]:
def make_preprocessor(numeric: list[str], categorical: list[str], scale_numeric: bool) -> ColumnTransformer:
    numeric_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])

def regression_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict:
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
        "within_10_pct": 100 * np.mean(np.abs(actual - predicted) <= 10),
    }

# For every horizon, preserve an unseen future period.
def metro_split(frame: pd.DataFrame, target: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    usable = frame[frame[target].notna()].copy()
    train = usable[usable.date < METRO_TEST_START].copy()
    test = usable[usable.date >= METRO_TEST_START].copy()
    assert train.date.max() < test.date.min()
    return train, test


## 4. Table 1 ? eight-model t+1 AQI benchmark

All models receive exactly the same leakage-safe t+1 predictors and the same chronological split. Training and inference timings are measured in the active kernel; report hardware and package versions with the paper.


In [ ]:
target_h1 = "target_aqi_h1"
train_h1, test_h1 = metro_split(metro, target_h1)
X_train, y_train = train_h1[METRO_NUMERIC + METRO_CATEGORICAL], train_h1[target_h1]
X_test, y_test = test_h1[METRO_NUMERIC + METRO_CATEGORICAL], test_h1[target_h1]

scaled_preprocessor = make_preprocessor(METRO_NUMERIC, METRO_CATEGORICAL, scale_numeric=True)
tree_preprocessor = make_preprocessor(METRO_NUMERIC, METRO_CATEGORICAL, scale_numeric=False)

benchmark_models = {
    "Ridge Regression": Pipeline([("prep", scaled_preprocessor), ("model", Ridge(alpha=10.0))]),
    "KNN": Pipeline([("prep", scaled_preprocessor), ("model", KNeighborsRegressor(n_neighbors=15, weights="distance", n_jobs=-1))]),
    "SVR": Pipeline([("prep", scaled_preprocessor), ("model", SVR(C=20.0, epsilon=0.1, gamma="scale"))]),
    "Random Forest": Pipeline([("prep", tree_preprocessor), ("model", RandomForestRegressor(
        n_estimators=TREE_ESTIMATORS, max_depth=20, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
    ))]),
    "Gradient Boosting": Pipeline([("prep", tree_preprocessor), ("model", GradientBoostingRegressor(
        n_estimators=TREE_ESTIMATORS, learning_rate=0.05, max_depth=3, loss="huber", random_state=RANDOM_STATE
    ))]),
    "XGBoost": Pipeline([("prep", tree_preprocessor), ("model", XGBRegressor(
        n_estimators=TREE_ESTIMATORS, max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE
    ))]),
}
if LIGHTGBM_AVAILABLE:
    benchmark_models["LightGBM"] = Pipeline([("prep", tree_preprocessor), ("model", LGBMRegressor(
        n_estimators=TREE_ESTIMATORS, learning_rate=0.05, num_leaves=31, subsample=0.85, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
    ))])
if CATBOOST_AVAILABLE:
    benchmark_models["CatBoost"] = Pipeline([("prep", tree_preprocessor), ("model", CatBoostRegressor(
        iterations=TREE_ESTIMATORS, depth=6, learning_rate=0.05, loss_function="RMSE",
        random_seed=RANDOM_STATE, verbose=False, thread_count=-1
    ))])

benchmark_results, benchmark_predictions, fitted_benchmarks = [], [], {}
for name, pipeline in benchmark_models.items():
    started = perf_counter()
    pipeline.fit(X_train, y_train)
    train_seconds = perf_counter() - started
    started = perf_counter()
    predicted = pipeline.predict(X_test)
    inference_seconds = perf_counter() - started
    row = {"model": name, **regression_metrics(y_test.to_numpy(), predicted),
           "train_seconds": train_seconds, "inference_ms_per_row": 1000 * inference_seconds / len(X_test)}
    benchmark_results.append(row)
    fitted_benchmarks[name] = pipeline
    result = test_h1[["date", "area", target_h1]].copy()
    result["model"] = name
    result["predicted_aqi"] = predicted
    benchmark_predictions.append(result)

# Make missing optional packages explicit rather than silently misrepresenting the benchmark.
for name, available in {"LightGBM": LIGHTGBM_AVAILABLE, "CatBoost": CATBOOST_AVAILABLE}.items():
    if not available:
        benchmark_results.append({"model": name, "status": "NOT RUN ? install package from setup cell"})

benchmark_table = pd.DataFrame(benchmark_results)
benchmark_table.to_csv(REPORT_DIR / "table_1_eight_model_benchmark.csv", index=False)
pd.concat(benchmark_predictions, ignore_index=True).to_csv(REPORT_DIR / "benchmark_t_plus_1_predictions.csv", index=False)
display(benchmark_table.sort_values("MAE", na_position="last").round(3))


## 5. Benchmark visualisation

In [ ]:
complete_benchmark = benchmark_table.dropna(subset=["MAE"]).sort_values("MAE")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=complete_benchmark, y="model", x="MAE", ax=axes[0], color="#2878B5")
axes[0].set_title("t+1 AQI forecast error (lower is better)")
axes[0].set_xlabel("MAE (AQI points)")
sns.barplot(data=complete_benchmark, y="model", x="R2", ax=axes[1], color="#2E8B57")
axes[1].set_title("t+1 AQI explained variance (higher is better)")
axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.savefig(REPORT_DIR / "table_1_benchmark_plot.png", dpi=160)
plt.show()


## 6. Multi-horizon t+1/t+2/t+3 AQI trajectory forecast

A `MultiOutputRegressor` wraps XGBoost to train one pure-ML estimator per horizon with a common leakage-safe feature table. The final test period remains completely later than model fitting.


In [ ]:
targets = [f"target_aqi_h{h}" for h in FORECAST_HORIZONS]
complete_horizons = metro.dropna(subset=targets).copy()
train_multi = complete_horizons[complete_horizons.date < METRO_TEST_START].copy()
test_multi = complete_horizons[complete_horizons.date >= METRO_TEST_START].copy()

base_xgb = XGBRegressor(
    n_estimators=TREE_ESTIMATORS, max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.8,
    min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
)
multi_horizon_model = Pipeline([
    ("prep", make_preprocessor(METRO_NUMERIC, METRO_CATEGORICAL, scale_numeric=False)),
    ("model", MultiOutputRegressor(base_xgb, n_jobs=1)),
])
started = perf_counter()
multi_horizon_model.fit(train_multi[METRO_NUMERIC + METRO_CATEGORICAL], train_multi[targets])
multi_train_seconds = perf_counter() - started
predicted_matrix = multi_horizon_model.predict(test_multi[METRO_NUMERIC + METRO_CATEGORICAL])
joblib.dump(multi_horizon_model, MODEL_DIR / "metro_multi_horizon_xgboost.joblib")

horizon_rows, horizon_predictions = [], []
for index, horizon in enumerate(FORECAST_HORIZONS):
    actual = test_multi[f"target_aqi_h{horizon}"].to_numpy()
    predicted = predicted_matrix[:, index]
    horizon_rows.append({"horizon_days": horizon, **regression_metrics(actual, predicted),
                         "train_seconds_for_joint_model": multi_train_seconds})
    item = test_multi[["date", "area", f"target_aqi_h{horizon}"]].copy()
    item["forecast_date"] = item.date + pd.Timedelta(days=horizon)
    item["horizon_days"] = horizon
    item["predicted_aqi"] = predicted
    horizon_predictions.append(item.rename(columns={f"target_aqi_h{horizon}": "actual_aqi"}))

horizon_table = pd.DataFrame(horizon_rows)
horizon_table.to_csv(REPORT_DIR / "multi_horizon_regression_metrics.csv", index=False)
pd.concat(horizon_predictions, ignore_index=True).to_csv(REPORT_DIR / "multi_horizon_regression_predictions.csv", index=False)
display(horizon_table.round(3))

plt.figure(figsize=(7, 4))
sns.lineplot(data=horizon_table, x="horizon_days", y="MAE", marker="o", linewidth=2)
plt.xticks(FORECAST_HORIZONS)
plt.title("Forecast-horizon decay curve")
plt.xlabel("Days ahead")
plt.ylabel("MAE (AQI points)")
plt.tight_layout()
plt.savefig(REPORT_DIR / "horizon_decay_curve.png", dpi=160)
plt.show()


## 6A. Required regression baseline comparisons

A paper must show that a machine-learning forecast beats simple, reproducible alternatives. This section compares multi-output XGBoost with:

- **Persistence:** AQI(t+h) = AQI at issue time t.
- **Area-month seasonal mean:** historical target AQI for the same area and known future month, estimated from training data only.

All three approaches use exactly the same unseen test rows.


In [ ]:
def area_month_baseline(train: pd.DataFrame, test: pd.DataFrame, target: str, horizon: int) -> np.ndarray:
    """Training-only area-and-forecast-month mean with area/global fallbacks."""
    train_lookup = train[["area", "date", target]].copy()
    test_lookup = test[["area", "date"]].copy()
    train_lookup["forecast_month"] = (train_lookup.date + pd.Timedelta(days=horizon)).dt.month
    test_lookup["forecast_month"] = (test_lookup.date + pd.Timedelta(days=horizon)).dt.month
    by_area_month = train_lookup.groupby(["area", "forecast_month"])[target].mean()
    by_area = train_lookup.groupby("area")[target].mean()
    global_mean = float(train_lookup[target].mean())
    return np.asarray([
        by_area_month.get((row.area, row.forecast_month), by_area.get(row.area, global_mean))
        for row in test_lookup.itertuples(index=False)
    ], dtype=float)

baseline_regression_rows = []
for position, horizon in enumerate(FORECAST_HORIZONS):
    target = f"target_aqi_h{horizon}"
    actual = test_multi[target].to_numpy()
    predictions = {
        "Persistence (AQI at t)": test_multi["aqi_at_issue"].to_numpy(),
        "Area-month seasonal mean": area_month_baseline(train_multi, test_multi, target, horizon),
        "Multi-output XGBoost": predicted_matrix[:, position],
    }
    for model_name, prediction in predictions.items():
        baseline_regression_rows.append({"horizon_days": horizon, "model": model_name, **regression_metrics(actual, prediction)})

baseline_regression_table = pd.DataFrame(baseline_regression_rows)
baseline_regression_table.to_csv(REPORT_DIR / "multi_horizon_baseline_comparison.csv", index=False)
display(baseline_regression_table.sort_values(["horizon_days", "MAE"]).round(3))

plt.figure(figsize=(9, 5))
sns.barplot(data=baseline_regression_table, x="horizon_days", y="MAE", hue="model")
plt.title("Forecasting baseline comparison on identical unseen test rows")
plt.xlabel("Forecast horizon (days)")
plt.ylabel("MAE (AQI points; lower is better)")
plt.tight_layout()
plt.savefig(REPORT_DIR / "multi_horizon_baseline_comparison.png", dpi=160)
plt.show()


## 7. Nationwide early-warning feature table

This section uses the 291 named-area dataset (292 state?area location keys). It deliberately excludes pollutant-indicator flags because they are derived from a same-day AQI reporting field. It uses current AQI at issue time and prior AQI/weather histories, then predicts future threshold exceedance.


In [ ]:
FULL_WEATHER = [c for c in WEATHER if c in pd.read_csv(FULL_PATH, nrows=1).columns]

def make_nationwide_event_table(path: Path) -> tuple[pd.DataFrame, list[str], list[str]]:
    data = pd.read_csv(path, parse_dates=["date"]).copy()
    data["location_key"] = data.state.astype(str).str.strip() + "||" + data.area.astype(str).str.strip()
    data = data.sort_values(["location_key", "date"]).drop_duplicates(["location_key", "date"]).reset_index(drop=True)
    group = data.groupby("location_key", sort=False)
    data = data.rename(columns={"aqi_value": "aqi_at_issue"})
    group = data.groupby("location_key", sort=False)

    for lag in (1, 3, 7, 14):
        col = f"aqi_lag_{lag}"
        data[col] = group["aqi_at_issue"].shift(lag)
        lag_date = group.date.shift(lag)
        data.loc[(data.date - lag_date).dt.days.ne(lag), col] = np.nan
    prior = group["aqi_at_issue"].shift(1)
    for window in (3, 7, 14):
        data[f"aqi_mean_{window}d"] = prior.groupby(data.location_key).transform(lambda s: s.rolling(window, min_periods=window).mean())
        data[f"aqi_max_{window}d"] = prior.groupby(data.location_key).transform(lambda s: s.rolling(window, min_periods=window).max())
        earliest = group.date.shift(window)
        invalid = (data.date - earliest).dt.days.ne(window)
        data.loc[invalid, [f"aqi_mean_{window}d", f"aqi_max_{window}d"]] = np.nan

    for weather_col in FULL_WEATHER:
        for lag in (1, 3, 7):
            col = f"{weather_col}_lag_{lag}"
            data[col] = group[weather_col].shift(lag)
            lag_date = group.date.shift(lag)
            data.loc[(data.date - lag_date).dt.days.ne(lag), col] = np.nan

    for horizon in FORECAST_HORIZONS:
        future_date = data.date + pd.Timedelta(days=horizon)
        data[f"forecast_month_h{horizon}"] = future_date.dt.month
        data[f"forecast_doy_sin_h{horizon}"] = np.sin(2 * np.pi * future_date.dt.dayofyear / 365.25)
        data[f"forecast_doy_cos_h{horizon}"] = np.cos(2 * np.pi * future_date.dt.dayofyear / 365.25)
        target = group.aqi_at_issue.shift(-horizon)
        target_date = group.date.shift(-horizon)
        data[f"target_aqi_h{horizon}"] = target.where((target_date - data.date).dt.days.eq(horizon))
        for cutoff in EVENT_THRESHOLDS:
            data[f"event_gt_{cutoff}_h{horizon}"] = (data[f"target_aqi_h{horizon}"] > cutoff).astype(float)
            data.loc[data[f"target_aqi_h{horizon}"].isna(), f"event_gt_{cutoff}_h{horizon}"] = np.nan

    numeric = ["aqi_at_issue", "latitude", "longitude"]
    numeric += [c for c in data if c.startswith(("aqi_lag_", "aqi_mean_", "aqi_max_"))]
    numeric += [c for c in data if any(c.startswith(f"{w}_lag_") for w in FULL_WEATHER)]
    numeric += [c for c in data if c.startswith(("forecast_month_", "forecast_doy_"))]
    return data, numeric, ["state", "location_key"]

nationwide, EW_NUMERIC, EW_CATEGORICAL = make_nationwide_event_table(FULL_PATH)
print(f"Nationwide rows: {len(nationwide):,}; named areas: {nationwide.area.nunique()}; state-area keys: {nationwide.location_key.nunique()}; feature count: {len(EW_NUMERIC) + len(EW_CATEGORICAL)}")


## 8. Cost-sensitive event classification for AQI >200 and AQI >300

Class weight is calculated using training data only. The probability threshold is selected on a separate validation period to target recall ? 0.80. The final metrics, ROC curves, precision-recall curves, and confusion matrices are calculated only on unseen future data.


In [ ]:
def select_threshold(y_true: pd.Series, probabilities: np.ndarray, target_recall: float = TARGET_RECALL) -> float:
    choices = []
    for threshold in np.arange(0.05, 0.96, 0.01):
        labels = probabilities >= threshold
        choices.append((threshold, recall_score(y_true, labels, zero_division=0), precision_score(y_true, labels, zero_division=0)))
    choice = pd.DataFrame(choices, columns=["threshold", "recall", "precision"])
    eligible = choice[choice.recall >= target_recall]
    if not eligible.empty:
        return float(eligible.sort_values(["precision", "threshold"], ascending=False).iloc[0].threshold)
    return float(choice.sort_values(["recall", "precision"], ascending=False).iloc[0].threshold)

def classification_summary(y_true: pd.Series, probability: np.ndarray, threshold: float) -> tuple[dict, np.ndarray]:
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "ROC_AUC": roc_auc_score(y_true, probability),
        "PR_AUC": average_precision_score(y_true, probability),
        "Recall": recall_score(y_true, prediction, zero_division=0),
        "Precision": precision_score(y_true, prediction, zero_division=0),
        "Balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }, prediction

def make_event_pipeline(scale_pos_weight: float) -> Pipeline:
    return Pipeline([
        ("prep", make_preprocessor(EW_NUMERIC, EW_CATEGORICAL, scale_numeric=False)),
        ("model", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", n_estimators=TREE_ESTIMATORS,
            max_depth=6, learning_rate=0.05, subsample=0.85, colsample_bytree=0.8,
            min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
            tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
        )),
    ])

event_rows, event_predictions = [], []
for horizon in FORECAST_HORIZONS:
    usable = nationwide[nationwide[f"target_aqi_h{horizon}"].notna()].copy()
    train = usable[usable.date <= EW_TRAIN_END]
    valid = usable[(usable.date > EW_TRAIN_END) & (usable.date <= EW_VALID_END)]
    test = usable[(usable.date > EW_VALID_END) & (usable.date <= EW_TEST_END)]
    for cutoff in EVENT_THRESHOLDS:
        label = f"event_gt_{cutoff}_h{horizon}"
        y_train, y_valid, y_test = train[label].astype(int), valid[label].astype(int), test[label].astype(int)
        if y_train.sum() == 0 or y_valid.nunique() < 2 or y_test.nunique() < 2:
            print(f"Skipped h={horizon}, AQI>{cutoff}: insufficient event variation")
            continue
        weight = ((len(y_train) - y_train.sum()) / y_train.sum()) * COST_MULTIPLIER
        model = make_event_pipeline(weight)
        started = perf_counter()
        model.fit(train[EW_NUMERIC + EW_CATEGORICAL], y_train)
        train_seconds = perf_counter() - started
        valid_prob = model.predict_proba(valid[EW_NUMERIC + EW_CATEGORICAL])[:, 1]
        alert_threshold = select_threshold(y_valid, valid_prob)
        test_prob = model.predict_proba(test[EW_NUMERIC + EW_CATEGORICAL])[:, 1]
        summary, alert = classification_summary(y_test, test_prob, alert_threshold)
        task = f"AQI_gt_{cutoff}_t_plus_{horizon}"
        joblib.dump(model, MODEL_DIR / f"{task}.joblib")
        event_rows.append({
            "task": task, "horizon_days": horizon, "AQI_cutoff": cutoff,
            "train_event_rate": y_train.mean(), "scale_pos_weight": weight,
            "probability_threshold": alert_threshold, "train_seconds": train_seconds, **summary,
        })
        output = test[["date", "state", "area", "location_key", f"target_aqi_h{horizon}"]].copy()
        output["forecast_date"] = output.date + pd.Timedelta(days=horizon)
        output["task"] = task
        output["event_probability"] = test_prob
        output["event_observed"] = y_test.to_numpy()
        output["alert_issued"] = alert.astype(int)
        event_predictions.append(output)

event_table = pd.DataFrame(event_rows).sort_values(["AQI_cutoff", "horizon_days"])
event_test_predictions = pd.concat(event_predictions, ignore_index=True)
event_table.to_csv(REPORT_DIR / "nationwide_early_warning_metrics.csv", index=False)
event_test_predictions.to_csv(REPORT_DIR / "nationwide_early_warning_test_predictions.csv", index=False)
display(event_table.round(3))


## 8A. Unweighted severe-event classifier baseline

The next comparison measures the effect of recall-oriented cost weighting. The unweighted XGBoost model uses `scale_pos_weight=1` and its default 0.50 probability threshold. It is compared with the cost-sensitive model whose alert threshold was selected on validation data.


In [ ]:
unweighted_rows = []
for horizon in FORECAST_HORIZONS:
    usable = nationwide[nationwide[f"target_aqi_h{horizon}"].notna()].copy()
    train = usable[usable.date <= EW_TRAIN_END]
    test = usable[(usable.date > EW_VALID_END) & (usable.date <= EW_TEST_END)]
    for cutoff in EVENT_THRESHOLDS:
        label = f"event_gt_{cutoff}_h{horizon}"
        y_train, y_test = train[label].astype(int), test[label].astype(int)
        if y_train.sum() == 0 or y_test.nunique() < 2:
            continue
        # No class weighting and no threshold tuning: transparent rare-event baseline.
        unweighted_model = make_event_pipeline(scale_pos_weight=1.0)
        started = perf_counter()
        unweighted_model.fit(train[EW_NUMERIC + EW_CATEGORICAL], y_train)
        train_seconds = perf_counter() - started
        probability = unweighted_model.predict_proba(test[EW_NUMERIC + EW_CATEGORICAL])[:, 1]
        summary, _ = classification_summary(y_test, probability, threshold=0.50)
        unweighted_rows.append({
            "variant": "Unweighted XGBoost (default 0.50 threshold)",
            "horizon_days": horizon, "AQI_cutoff": cutoff, "scale_pos_weight": 1.0,
            "probability_threshold": 0.50, "train_seconds": train_seconds, **summary,
        })

unweighted_event_table = pd.DataFrame(unweighted_rows)
cost_sensitive_comparison = event_table.copy()
cost_sensitive_comparison["variant"] = "Cost-sensitive XGBoost (validation-selected threshold)"
comparison_columns = ["variant", "horizon_days", "AQI_cutoff", "scale_pos_weight", "probability_threshold", "ROC_AUC", "PR_AUC", "Recall", "Precision", "Balanced_accuracy", "TP", "FP", "FN", "TN"]
event_baseline_comparison = pd.concat([
    cost_sensitive_comparison[comparison_columns], unweighted_event_table[comparison_columns]
], ignore_index=True)
event_baseline_comparison.to_csv(REPORT_DIR / "event_unweighted_baseline_comparison.csv", index=False)
display(event_baseline_comparison.sort_values(["AQI_cutoff", "horizon_days", "variant"]).round(3))

plot = event_baseline_comparison.melt(
    id_vars=["variant", "horizon_days", "AQI_cutoff"], value_vars=["Recall", "Precision"],
    var_name="metric", value_name="score"
)
g = sns.catplot(data=plot, x="horizon_days", y="score", hue="variant", col="AQI_cutoff", row="metric", kind="bar", height=3.5, aspect=1.2)
g.set_axis_labels("Forecast horizon (days)", "Score")
g.fig.tight_layout()
g.fig.savefig(REPORT_DIR / "event_weighting_baseline_comparison.png", dpi=160)
plt.show()


## 9. Early-warning figures, confusion matrices, and example alert feed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=event_table, x="task", y="Recall", ax=axes[0], color="#C44E52")
axes[0].set_ylim(0, 1.05); axes[0].tick_params(axis="x", rotation=25); axes[0].set_title("Hazard-event recall")
sns.barplot(data=event_table, x="task", y="Precision", ax=axes[1], color="#4C72B0")
axes[1].set_ylim(0, 1.05); axes[1].tick_params(axis="x", rotation=25); axes[1].set_title("Alert precision")
plt.tight_layout(); plt.savefig(REPORT_DIR / "early_warning_recall_precision.png", dpi=160); plt.show()

for task in event_table.task:
    sample = event_test_predictions[event_test_predictions.task.eq(task)]
    matrix = confusion_matrix(sample.event_observed, sample.alert_issued, labels=[0, 1])
    plt.figure(figsize=(4, 3.5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=["No alert", "Alert"], yticklabels=["No event", "Event"])
    plt.title(f"Confusion matrix ? {task}")
    plt.ylabel("Observed"); plt.xlabel("Issued")
    plt.tight_layout(); plt.savefig(REPORT_DIR / f"confusion_{task}.png", dpi=160); plt.show()

# ROC and precision-recall curves use continuous event probabilities.
for task in event_table.task:
    sample = event_test_predictions[event_test_predictions.task.eq(task)]
    fpr, tpr, _ = roc_curve(sample.event_observed, sample.event_probability)
    precision, recall, _ = precision_recall_curve(sample.event_observed, sample.event_probability)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(fpr, tpr, color="#C44E52")
    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title=f"ROC ? {task}")
    axes[1].plot(recall, precision, color="#4C72B0")
    axes[1].set(xlabel="Recall", ylabel="Precision", title=f"Precision-recall ? {task}")
    plt.tight_layout(); plt.savefig(REPORT_DIR / f"roc_pr_{task}.png", dpi=160); plt.show()

alert_feed = event_test_predictions[event_test_predictions.alert_issued.eq(1)].copy()
alert_feed["risk_message"] = np.where(
    alert_feed.task.str.contains("gt_300"),
    "Severe-smog early warning: forecast AQI >300",
    "Health-risk early warning: forecast AQI >200",
)
alert_feed = alert_feed.sort_values(["forecast_date", "event_probability"], ascending=[False, False])
alert_feed.to_csv(REPORT_DIR / "nationwide_issued_alerts_test_period.csv", index=False)
display(alert_feed[["date", "forecast_date", "state", "area", "task", "event_probability", "event_observed", "risk_message"]].head(25))


## 10. Exported research artefacts and reporting guardrails

The notebook creates Table 1, horizon-decay metrics, predictions, six early-warning models, figures, confusion matrices, and alert feeds. Use the generated files?not hard-coded values?in the manuscript.

For a publishable paper, also report confidence intervals (e.g. date-block bootstrap), state/region subgroup results, per-horizon event prevalence, hardware/package versions, hyperparameter-selection protocol, and the impact of missing weather/geocoding data. Never present reanalysis back-testing as a live forecast experiment.


In [ ]:
run_manifest = {
    "metro_input": str(METRO_PATH.relative_to(PROJECT_ROOT)),
    "nationwide_input": str(FULL_PATH.relative_to(PROJECT_ROOT)),
    "models_requested": ["Ridge", "KNN", "SVR", "Random Forest", "Gradient Boosting", "LightGBM", "CatBoost", "XGBoost"],
    "horizons": FORECAST_HORIZONS, "event_thresholds": EVENT_THRESHOLDS,
    "metro_test_start": str(METRO_TEST_START.date()),
    "early_warning_train_end": str(EW_TRAIN_END.date()),
    "early_warning_validation_end": str(EW_VALID_END.date()),
    "early_warning_test_end": str(EW_TEST_END.date()),
    "tree_estimators": TREE_ESTIMATORS, "quick_run": QUICK_RUN,
    "target_validation_recall": TARGET_RECALL, "cost_multiplier": COST_MULTIPLIER,
}
with (REPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as file:
    json.dump(run_manifest, file, indent=2)

print("Report files:")
for path in sorted(REPORT_DIR.iterdir()):
    print(" -", path.relative_to(PROJECT_ROOT))
print(f"\nModel files written: {len(list(MODEL_DIR.glob('*.joblib')))}")
